Hay que evaluar la performance de la inferencia. 

In [6]:
import pandas as pd
from pathlib import Path
from taed2_safeagent.data.inputs import build_input


# Cargar los comandos desde tu dataset limpio o de evaluación
#project_root = Path(__file__).resolve().parents[2]
project_root = Path.cwd().parents[1]
print(f"Project root: {project_root}")

data_path = project_root / "data/processed/shell_safety/test.jsonl"
df = pd.read_json(data_path, lines=True)

sample_rows = df[:30].to_dict(orient="records")

sample_inputs = {
    "command": [build_input(row, "command") for row in sample_rows],
    "command-context": [build_input(row, "command-context") for row in sample_rows],
}

Project root: c:\Users\Usuario\Desktop\taed2-SafeAgent


In [ ]:
''' INFERENCIA PARA EL BASELINE '''

import joblib
model_dir = project_root / "models/tfidf_char_linear_svm"

pipeline = joblib.load(model_dir / "pipeline.joblib")
predicted_labels = pipeline.predict(sample_inputs["command"])

predicted_labels


array(['ALLOW', 'ASK', 'ASK', 'ALLOW', 'ALLOW', 'ASK', 'ASK', 'ASK',
       'DENY', 'ALLOW', 'DENY', 'ALLOW', 'ALLOW', 'ASK', 'ASK', 'ALLOW',
       'ALLOW', 'ALLOW', 'ASK', 'ASK', 'ALLOW', 'ASK', 'ASK', 'ALLOW',
       'DENY', 'ALLOW', 'DENY', 'DENY', 'ALLOW', 'ASK'], dtype='<U5')

In [ ]:
''' INFERENCIA DE MODELOS COMPLEJOS '''


import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = project_root / "models/encoders/codebert__command__2__a740d875"

# 1. Cargar tokenizer y modelo en CPU
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to("cpu")
model.eval()

# 2. Tokenizar la lista de textos
inputs = tokenizer(
    sample_inputs["command"],
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
).to("cpu")

# 3. Inferencia
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).tolist()

predicted_labels = [model.config.id2label[i] for i in predictions]


In [ ]:
''' Calcular métricas de CPU para modelos candidate '''
import time
import numpy as np

# Model size: Tamaño en MB del modelo
model_dir = project_root / "models/encoders/codebert__command__2__a740d875"
model_size_mb = sum(
    f.stat().st_size for f in model_dir.rglob("*") if f.is_file()
) / (1024 * 1024)


# Load time: Tiempo que tarda en cargar el modelo en memoria
start = time.perf_counter()
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
load_time = time.perf_counter() - start


# Repetition count: Repeticiones de todo el proceso de hacer las len(sample_inputs) inferencias. 
repetition_count = 300


# Median latency and Latency p95: 
latencies_list = [] #Es una lista que guarda cuánto ha tardado cada inferencia en ms
median_ms = np.median(latencies_list)
p95_ms = np.percentile(latencies_list, 95)
